# 04 - Ingestão Bronze Unificada: Pedidos (Hash) e Clientes (ADLS)
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** `ecommerce_pedidos` (Missão 01) e `ecommerce_clientes`  
**Branch:** `feat/squad2/kauan-leandro`  

### Objetivo da Task (Bronze Unificada):

**Parte 1 — Missão 01: ecommerce_pedidos (funciona no serverless):**
1. **Leitura da Origem:** Ler `ecommerce_pedidos` via Lakehouse Federation (SQL Server).
2. **Controle de Hash:** Calcular hash SHA-256 de cada pedido para detectar duplicatas e alterações.
3. **Ingestão Incremental:** Filtrar apenas pedidos novos ou alterados via anti-join com a tabela de controle.
4. **Persistência Bronze:** Append incremental em `pedidos_filtrados` + MERGE em `controle_pedidos`.
5. **Idempotência:** Reexecução não duplica registros já processados.

**Parte 2 — ecommerce_clientes (requer cluster clássico para ADLS):**
1. **Leitura da Origem:** Ler arquivos `ecommerce_clientes.parquet` do ADLS Gen2.
2. **Detecção Incremental:** Identificar arquivos novos via anti-join com o log de controle.
3. **Persistência Bronze:** Gravar no ADLS Delta (append) com `bronze_ingested_at`, particionado por ano/mes/dia/hora.
4. **Monitoramento:** Loop de verificacão a cada 15 segundos (max 90s) com tolerância a falhas.

> **⚠️ Nota serverless:** A Parte 1 (pedidos via Federation) funciona no serverless. A Parte 2 (clientes via ADLS) requer cluster clássico. No serverless, a Silver de clientes lê direto da origem via Federation.

## KPIs e Tabelas Geradas

| Tabela UC | Função | Modo |
| --- | --- | --- |
| `workspace.default.controle_pedidos` | Rastreia hash SHA-256 de cada pedido | Upsert (MERGE) |
| `workspace.default.pedidos_filtrados` | Pedidos novos/alterados (Bronze) | Append incremental |

In [0]:
%run ./00_setup_config.ipynb

In [0]:
# Importação das bibliotecas necessárias (Bronze Unificada)

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import os

# Bibliotecas ADLS (Parte 2 — clientes, requer cluster clássico)
try:
    from azure.identity import ClientSecretCredential
    from azure.storage.filedatalake import DataLakeServiceClient
    from dotenv import load_dotenv
    from pathlib import Path
    import pandas as pd
    from io import BytesIO
    AZURE_SDK_OK = True
except ImportError:
    AZURE_SDK_OK = False
    print("⚠️ Azure SDK não disponível (serverless) — Parte 2 (clientes ADLS) será pulada")

---
# Parte 1 — Missão 01: ecommerce_pedidos (Hash SHA-256)

Funciona no serverless via Lakehouse Federation. Cria tabela de controle com hash SHA-256 de cada pedido para detectar duplicatas e alterações.

In [0]:
# ============================================================================
# Configuracao de camadas: ADLS (primario) + Unity Catalog (fallback serverless)
# ============================================================================

storage_account_name = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

tabela_origem_pedidos = "sqlserver_catalog.squad2.ecommerce_pedidos"
tabela_controle = "workspace.default.controle_pedidos"
tabela_pedidos_filtrados = "workspace.default.pedidos_filtrados"

try:
    spark.read.format("parquet").options(**adls_options).load(
        f"abfss://raw@{storage_account_name}.dfs.core.windows.net/"
    ).limit(1).count()
    usar_adls = True
    print("ADLS acessivel")
except Exception:
    usar_adls = False
    print("ADLS bloqueado (serverless) — usando UC + Lakehouse Federation")

print(f"Origem pedidos: {tabela_origem_pedidos if not usar_adls else 'ADLS'}")

## Tabela de Controle

Cria `workspace.default.controle_pedidos` que rastreia o hash SHA-256 de cada pedido.

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.controle_pedidos (
        id_pedido           BIGINT,
        hash_registro       STRING,
        dt_processamento    TIMESTAMP
    )
    USING DELTA
    COMMENT 'Controle: rastreia hash SHA-256 de cada pedido processado.'
""")

print("Tabela de controle verificada/criada: workspace.default.controle_pedidos")

## Leitura da Origem

Le `ecommerce_pedidos` via Lakehouse Federation (serverless) ou ADLS (cluster classico).

In [0]:
if usar_adls:
    raise NotImplementedError("ADLS: definir leitura dos parquets de ecommerce_pedidos")
else:
    df_pedidos = spark.table(tabela_origem_pedidos)

print(f"Total de pedidos na origem: {df_pedidos.count()}")
display(df_pedidos.limit(5))

## Calculo do Hash SHA-256

Calcula o hash de todas as colunas (exceto `id_pedido`) para detectar alteracoes.

In [0]:
colunas_hash = [c for c in df_pedidos.columns if c != "id_pedido"]

expr_hash = F.sha2(
    F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in colunas_hash]),
    256
)

df_com_hash = df_pedidos.withColumn("hash_registro", expr_hash)

print(f"Hash calculado para {len(colunas_hash)} colunas.")

## Comparacao com Controle

Compara o hash atual com o hash armazenado: `hash_antigo IS NULL` (novo) ou `hash_novo != hash_antigo` (alterado).

In [0]:
df_controle = spark.table(tabela_controle)

df_diff = df_com_hash.alias("src").join(
    df_controle.alias("ctl"),
    F.col("src.id_pedido") == F.col("ctl.id_pedido"),
    "left"
).select(
    F.col("src.id_pedido"),
    F.col("src.hash_registro").alias("hash_novo"),
    F.col("ctl.hash_registro").alias("hash_antigo"),
)

df_novos = df_diff.filter(
    F.col("hash_antigo").isNull() | (F.col("hash_novo") != F.col("hash_antigo"))
)

total_novos = df_novos.count()
total_origem = df_com_hash.count()

print(f"Total na origem: {total_origem}")
print(f"Novos/alterados: {total_novos}")
print(f"Ja processados : {total_origem - total_novos}")

## Persistencia Bronze

`pedidos_filtrados` (append) + `controle_pedidos` (MERGE upsert).

In [0]:
if total_novos > 0:
    df_pedidos_novos = df_com_hash.join(df_novos.select("id_pedido"), "id_pedido", "inner")
    df_pedidos_novos.write.mode("append").saveAsTable(tabela_pedidos_filtrados)
    print(f"{total_novos} pedidos inseridos em {tabela_pedidos_filtrados}")
    
    df_atualizacao = df_novos.select(
        F.col("id_pedido"),
        F.col("hash_novo").alias("hash_registro"),
        F.current_timestamp().alias("dt_processamento")
    )
    df_atualizacao.createOrReplaceTempView("vw_atualizacao_pedidos")
    
    spark.sql("""
        MERGE INTO workspace.default.controle_pedidos AS ctl
        USING vw_atualizacao_pedidos AS src
        ON ctl.id_pedido = src.id_pedido
        WHEN MATCHED THEN UPDATE SET
            ctl.hash_registro = src.hash_registro,
            ctl.dt_processamento = src.dt_processamento
        WHEN NOT MATCHED THEN INSERT (id_pedido, hash_registro, dt_processamento)
        VALUES (src.id_pedido, src.hash_registro, src.dt_processamento)
    """)
    print(f"Controle atualizado: {tabela_controle}")
else:
    print("Nenhum pedido novo ou alterado.")

## Resumo da Parte 1 (Pedidos)

In [0]:
print("=== RESUMO BRONZE — ecommerce_pedidos ===")
print(f"Origem          : {tabela_origem_pedidos}")
print(f"Total na origem : {total_origem}")
print(f"Novos/alterados : {total_novos}")
print(f"Ja processados  : {total_origem - total_novos}")
try:
    total_acumulado = spark.table(tabela_pedidos_filtrados).count()
    print(f"Acumulado em {tabela_pedidos_filtrados}: {total_acumulado}")
except:
    print("Tabela filtrados ainda nao criada.")

---
# Parte 2 — ecommerce_clientes (ADLS Gen2)

> **⚠️ Requer cluster clássico** — Azure SDK não funciona no serverless. No serverless, a Silver de clientes le direto da origem via Lakehouse Federation (notebook 05).

In [0]:
# Cria a credencial para autenticar o acesso ao Azure

credential = ClientSecretCredential(
    tenant_id=os.getenv("ADLS_TENANT_ID"),
    client_id=os.getenv("ADLS_CLIENT_ID"),
    client_secret=os.getenv("ADLS_CLIENT_SECRET")
)

print("Credencial do Azure criada com sucesso.")

In [0]:
# Cria a conexão com a conta e o container do Data Lake

storage_account_name = "internshipdatalake"
container_name = "raw"

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

fs_client = service_client.get_file_system_client(container_name)

print("Conexão com o ADLS criada com sucesso.")

In [0]:
# Localiza os arquivos Parquet da ecommerce_clientes no Data Lake

arquivos_encontrados = []

for arquivo in fs_client.get_paths(path="real-time-data", recursive=True):
    if not arquivo.is_directory and arquivo.name.endswith("ecommerce_clientes.parquet"):
        arquivos_encontrados.append(arquivo.name)

print(f"Arquivos encontrados: {len(arquivos_encontrados)}")

for arquivo in arquivos_encontrados:
    print(arquivo)

In [0]:
# Prepara as credenciais OAuth para o Spark acessar o ADLS

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

print("Credenciais OAuth do Spark preparadas.")

In [0]:
adls_tenant_id = os.getenv("ADLS_TENANT_ID")
adls_client_id = os.getenv("ADLS_CLIENT_ID")
adls_client_secret = os.getenv("ADLS_CLIENT_SECRET")
adls_storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

import pandas as pd
from io import BytesIO

credential = ClientSecretCredential(adls_tenant_id, adls_client_id, adls_client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{adls_storage_account}.dfs.core.windows.net/",
    credential=credential,
)
file_system_client = service_client.get_file_system_client(file_system="raw")

print("Conexão via SDK pronta (file_system_client).")

In [0]:

caminho_bronze = (
    f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/"
    "bronze/ecommerce_clientes")
caminho_control_log = f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/grupo2/metadata/ingestion_control_log"

print(f"Caminho Bronze: {caminho_bronze}")
print(f"Caminho log de controle: {caminho_control_log}")

In [0]:
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("Função de verificação de tabela Delta pronta.")

In [0]:
def detectar_novos_clientes():
    todos_arquivos = [
        p.name for p in file_system_client.get_paths(path="real-time-data", recursive=True)
        if p.name.endswith("ecommerce_clientes.parquet")
    ]

    if tabela_delta_existe(caminho_control_log):
        df_control = spark.read.format("delta").options(**adls_options).load(caminho_control_log)
        ja_processados = {
            r["arquivo_processado"] for r in
            df_control.select("arquivo_processado").collect()
        }
    else:
        ja_processados = set()

    arquivos_novos = [a for a in todos_arquivos if a not in ja_processados]

    if not arquivos_novos:
        return None, []

    dfs = []
    for caminho in arquivos_novos:
        file_client = file_system_client.get_file_client(caminho)
        conteudo = file_client.download_file().readall()
        pdf = pd.read_parquet(BytesIO(conteudo))
        pdf["bronze_source_file"] = caminho 
        dfs.append(pdf)

    pdf_final = pd.concat(dfs, ignore_index=True)
    df_novos = spark.createDataFrame(pdf_final)

    return df_novos, arquivos_novos

print("Função de detecção de arquivos novos pronta.")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import current_timestamp, year, month, dayofmonth, hour, col, count
from datetime import datetime

schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False),
    StructField("timestamp_processamento", StringType(), False),
])

def processar_bronze(df_novos, arquivos_novos):
    if df_novos is None:
        print("    Nenhum arquivo novo para 'ecommerce_clientes'.")
        return

    total = df_novos.count()
    print(f"    >>> {len(arquivos_novos)} arquivo(s) novo(s) detectado(s) ({total} linhas)")

    df_bronze = (df_novos
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("ano", year(col("bronze_ingested_at")))
        .withColumn("mes", month(col("bronze_ingested_at")))
        .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
        .withColumn("hora", hour(col("bronze_ingested_at"))))

    (df_bronze.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminho_bronze))

    print("    [OK] Bronze gravada.")

    try:
        resumo_por_arquivo = (df_novos
            .groupBy("bronze_source_file")
            .agg(count("*").alias("linhas"))
            .collect())

        agora = str(datetime.now())
        log_data = [("ecommerce_clientes", r["bronze_source_file"], int(r["linhas"]), "SUCCESS", agora) for r in resumo_por_arquivo]
        df_log = spark.createDataFrame(log_data, schema=schema_log)

        (df_log.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminho_control_log))

        print("    [OK] Log de controle atualizado.")
    except Exception as e_log:
        print(f"    [ERRO NO LOG] {type(e_log).__name__}: {e_log}")

print("Função de gravação na Bronze pronta.")

In [0]:
import time
from datetime import datetime, timedelta

intervalo_segundos = 15
duracao_maxima_segundos = 90
inicio = datetime.now()
fim_previsto = inicio + timedelta(seconds=duracao_maxima_segundos)
falhas_consecutivas = 0
MAX_FALHAS_CONSECUTIVAS = 5

print(f"Iniciando monitoramento às {inicio.strftime('%H:%M:%S')}.")
print(f"Vai rodar até {fim_previsto.strftime('%H:%M:%S')} (ou até interromper manualmente).\n")

while datetime.now() < fim_previsto:
    try:
        print(f"--- Verificação às {datetime.now().strftime('%H:%M:%S')} ---")
        df_novos, arquivos_novos = detectar_novos_clientes()
        processar_bronze(df_novos, arquivos_novos)
        falhas_consecutivas = 0
    except KeyboardInterrupt:
        print("\nMonitoramento interrompido manualmente.")
        break
    except Exception as e:
        falhas_consecutivas += 1
        print(f"    [ERRO] Falha na verificação ({falhas_consecutivas}/{MAX_FALHAS_CONSECUTIVAS}): {e}")
        if falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
            print(f"    {MAX_FALHAS_CONSECUTIVAS} falhas seguidas — interrompendo o loop.")
            break
    time.sleep(intervalo_segundos)

print(f"\nMonitoramento finalizado às {datetime.now().strftime('%H:%M:%S')}.")